### เปรียบเทียบประสิทธิภาพ LLM ในการตรวจจับ Scam (Qwen, GLM)

In [4]:
import os
import pandas as pd
from openrouter import OpenRouter
from dotenv import load_dotenv  

load_dotenv()
api_key=os.getenv("OPENROUTER_API_KEY")

In [ ]:
SYSTEM_PROMPT = """
Role: คุณเป็นผู้เชี่ยวชาญด้านการวิเคราะห์ SMS หลอกลวง ในประเทศไทย และเป็นหน่วยรักษาความปลอดภัยทางไซเบอร์มานานกว่า 50 ปี

Task: ช้อความต่อไปนี้เป็นข้อความที่ส่งมาจาก sms ซึ่งใน ข้อความเหล่านี้จะมีทั้งข้อความปกติ และ ข้อความจากแสกมเมอร์ 
หน้าที่ของคุณคือวิเคราะห์ว่า sms เป็นข้อความปกติ หรือ ข้อความจากแสกมเมอร์ แล้วตอบมาแค่เป็นคำว่า "ปกติ" หรือ "แสกมเมอร์" เท่านั้น
ห้ามตอบอย่างอื่น นอกจากคำว่า "ปกติ" หรือ "แสกมเมอร์"
"""
    
def qwen_analysis (text):
    with OpenRouter(api_key=api_key) as client:
        response = client.chat.send(
            model="qwen/qwen3-32b",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": text},
            ]
        )

    return response.choices[0].message.content
def glm_analysis (text):
    with OpenRouter(api_key=api_key) as client:
        response = client.chat.send(
            model="z-ai/glm-4.5-air",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": text},
            ]
        )

    return response.choices[0].message.content

In [16]:
test_messages = [
    #Scam messages
    "ธนาคารได้อนุมัติวงเงิน50,000บาทติดต่อรับสิทธิ์: https://rb.gy/2erksz",
    "คุณได้รับคืนเงินประกันมิเตอร์ไฟฟ้าส่วนภูมิภาค PEA ซึ่งจะคืนตามขนาดของมิเตอร์ที่ใช้ สามารถสอบถามเพิ่มเติมได้ที่ https://bit.ly/44jHSpi",
    "KMA ยื่นสิทธิ์การกู้ให้ท่าน 120,000 บาท คลิก: citly.me/eq9br ",
    "เงินเข้าโทรศัพท์ฟรี 500บาท กด*158# ลุ้นก่อนใคร สมัครดวงดีดวงเฮง 3บ/ขค",
    "รับฟรี 129 บาท ฝากแรกรับ 1599 บาท",
    "ศูนย์รักษาความปลอดภัย ttb: พบข้อมูลบัญชีของท่านรั่วไหลจากแพลตฟอร์มช้อปปิ้งออนไลน์ กรุณาเปลี่ยนรหัส PIN ด่วนที่ https://secure.ttb-touch.com",
    "สวัสดีค่ะ เนื่องจากคุณเป็นผู้ใช้ที่มีประวัติที่ดี  ยินดีด้วยคุณถูกTikTok เลือกให้ทำงานหารายได้เสริมทางออนไลน์ ทำง่ายๆ จ่ายสดทุกบิล รายได้ต่อวัน3000บาท/วัน ติดต่อเรา@LINE：a3633",
    "J&T Express สวัสดีค่ะคุณมีประวัติการใช้บริการเกิน 5 ครั้ง ทาง สคร. จึงมอบพัดลมมินิเป็นของขวัญปีใหม่เพื่อเป็นการขอบคุณเพิ่ม ไลน์: sep298",
    "เข้าร่วมกลุ่มเพื่อผลตอบแทนที่มากขึ้น ติดต่อ LINE：  reurl.cc/G4qLkZ",
    "เครดิตฟรีรับง่าย ไม่ต้องแชร์ แค๊ปรูปมารับได้เลย คลิ๊ก bit.ly/43sTBRx",  
    #Normal messages
    "AIS ยกขบวนมือถือรุ่นเด็ด พร้อมโปรโมชันส่วนลดจัดเต็มในงาน Thailand Mobile Expo ณ ศูนย์การประชุมแห่งชาติสิริกิติ์ ระหว่างวันที่ 8-11 กุมภาพันธ์ 2567 เตรียมมาชอปมือถือใหม่ รับปีมังกร พร้อมลดภาษีได้เต็ม Max (ขึ้นอยู่กับอัตราเงินได้สุทธิที่เสียภาษี) แล้วพบกันภายในงานตั้งแต่ 10.00 น. – 20.00 น.",
    "OTP 930917 (ref. 8697) สำหรับสมัคร myID อย่าบอกรหัสนี้กับบุคคลอื่น",
    "ใจฟู เน็ตฟูลสปีด! เล่นไม่สะดุด 50วัน 199บ. 30วัน ถึง 29 ก.พ.67 คลิก https://dtac.co.th/s/CMTDTE5",
    "[ ⬛️ Express] พัสดุ ⬛️⬛️ จัดส่งสำเร็จแล้ว ตรวจสอบได้ที่เว็บไซต์ ⬛️ Express Thailandหากมีข้อสงสัย กรุณาติดต่อ call center 1470",
    "มีผู้เข้าสู่ระบบธนาคารของคุณจากอุปกรณ์อื่น หากไม่ได้ดำเนินการด้วยตนเอง โปรดติดต่อทันที kasikorn.go-line.cc",
    "ปีนี้ก็ยังแจกต่อ แจกฟรี SamsungGalaxy Z Flip5 เฉพาะ 6 ม.ค. 67 เล่นฟรีที่ทรูไอดี คลิก! https://bit.ly/3RstaZu",
    "ดีแทคห่วงใยนักช้อป ช้อปปิ้งออนไลน์แบบไร้กังวล ซื้อของไม่ใด้ของก็เคลมได้ ลดพิเศษถึง10% ใส่โค้ด Cyber ที่ดีชัวรันส์https://www.dtac.co.th/s/PQi6TCO",
    "Shopee 3.3 ซิมเน็ตรายปี เฉลี่ยเดือนละ 94 บาท https://shope.ee/5fQghO6syv (ยกเลิกข่าวสาร เข้าแอปฯ>ฉัน>ตั้งค่าบัญชี>ตั้งค่าการแจ้งเตือน)",
    "แจกกระหน่ำ ทุกวันอาทิตย์ ฟรีSamsung Galaxy S23 Ultra เฉพาะ 21ม.ค. 67 เล่นฟรี คลิก! https://ttid.co/ln4U/wf3dnlk3",
    "พิเศษ!สมาชิกSF Plus รับส่วนลด 50 .-Junji ito@MBKชั้น4 ถึง 30พ.ย.นี้"
    ]

results = []
for msg in test_messages:
    results.append({
        "Message": msg,
        "qwen3-32b ": qwen_analysis(msg),
        "glm-4.5-air": glm_analysis(msg)
    })

df_results = pd.DataFrame(results)
display(df_results)

,Message,qwen3-32b,glm-4.5-air
0,"ธนาคารได้อนุมัติวงเงิน50,000บาทติดต่อรับสิทธิ์: https://rb.gy/2erksz",แสกมเมอร์,แสกมเมอร์
1,คุณได้รับคืนเงินประกันมิเตอร์ไฟฟ้าส่วนภูมิภาค PEA ซึ่งจะคืนตามขนาดของมิเตอร์ที่ใช้ สามารถสอบถามเพิ่มเติมได้ที่ https://bit.ly/44jHSpi,แสกมเมอร์,แสกมเมอร์
2,"KMA ยื่นสิทธิ์การกู้ให้ท่าน 120,000 บาท คลิก: citly.me/eq9br",แสกมเมอร์,แสกมเมอร์
3,เงินเข้าโทรศัพท์ฟรี 500บาท กด*158# ลุ้นก่อนใคร สมัครดวงดีดวงเฮง 3บ/ขค,แสกมเมอร์,แสกมเมอร์
4,รับฟรี 129 บาท ฝากแรกรับ 1599 บาท,แสกมเมอร์,แสกมเมอร์
5,ศูนย์รักษาความปลอดภัย ttb: พบข้อมูลบัญชีของท่านรั่วไหลจากแพลตฟอร์มช้อปปิ้งออนไลน์ กรุณาเปลี่ยนรหัส PIN ด่วนที่ https://secure.ttb-touch.com,แสกมเมอร์,แสกมเมอร์
6,สวัสดีค่ะ เนื่องจากคุณเป็นผู้ใช้ที่มีประวัติที่ดี ยินดีด้วยคุณถูกTikTok เลือกให้ทำงานหารายได้เสริมทางออนไลน์ ทำง่ายๆ จ่ายสดทุกบิล รายได้ต่อวัน3000บาท/วัน ติดต่อเรา@LINE：a3633,แสกมเมอร์,แสกมเมอร์
7,J&T Express สวัสดีค่ะคุณมีประวัติการใช้บริการเกิน 5 ครั้ง ทาง สคร. จึงมอบพัดลมมินิเป็นของขวัญปีใหม่เพื่อเป็นการขอบคุณเพิ่ม ไลน์: sep298,แสกมเมอร์,แสกมเมอร์
8,เข้าร่วมกลุ่มเพื่อผลตอบแทนที่มากขึ้น ติดต่อ LINE： reurl.cc/G4qLkZ,แสกมเมอร์,แสกมเมอร์
9,เครดิตฟรีรับง่าย ไม่ต้องแชร์ แค๊ปรูปมารับได้เลย คลิ๊ก bit.ly/43sTBRx,แสกมเมอร์,NaN
